# Classification

In [48]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier
import joblib

pd.set_option('display.max_columns', None)

In [49]:
import warnings
from sklearn.exceptions import ConvergenceWarning, FitFailedWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=FitFailedWarning)


In [50]:
df_train = pd.read_csv('df_train.csv')
df_test = pd.read_csv('df_test.csv')

In [51]:
df_train.shape

(574, 302)

In [52]:
df_test.shape

(384, 302)

In [53]:
model_path = '/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/Modelli a tre anni con city/'

In [54]:
df_train['target'].value_counts()

target
2    401
1    119
0     54
Name: count, dtype: int64

1. Selezione anni di interesse

In [55]:
cols_to_drop = df_train.filter(regex=r"2020").columns.tolist()

In [56]:
df_train = df_train.drop(cols_to_drop, axis = 1)

In [57]:
df_test = df_test.drop(cols_to_drop, axis = 1)

In [58]:
df_train.shape

(574, 293)

In [59]:
## Creiamo un dizionario dove inserire i risultati della validazione
evaluation_results = {}

In [60]:
## VALORE DELLA CROSS VALIDATION
cv = 5

2. Divisione in train e validation

In [61]:
y = df_train['target']
X = df_train.drop('target', axis = 1)

In [62]:
x_train, x_val, y_train, y_val = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)

In [63]:
df_train.columns

Index(['works_count_2023', 'cited_by_count_2023', 'works_count_2022',
       'cited_by_count_2022', 'works_count_2021', 'cited_by_count_2021',
       'tot_dipendenti_per_anno_2021', 'studiosi_usciti_2021',
       'studiosi_entrati_2021', 'tot_dipendenti_per_anno_2022',
       ...
       'city_Vercelli', 'city_Verona', 'city_Vicenza', 'city_Vietri sul Mare',
       'city_Villa Guardia', 'city_Vimercate', 'city_Viterbo',
       'city_Vittorio Veneto', 'city_Vizzolo Predabissi', 'target'],
      dtype='object', length=293)

3. Scalatura dei dati

In [64]:
cols_to_scale = df_train.filter(regex=r"\d{4}").columns.tolist()
scaler = StandardScaler()
x_train[cols_to_scale] = scaler.fit_transform(x_train[cols_to_scale])
x_val[cols_to_scale] = scaler.transform(x_val[cols_to_scale])
df_test[cols_to_scale] = scaler.transform(df_test[cols_to_scale])

In [65]:
x_train.shape

(344, 292)

3. Validation Phase

In [66]:
svc = SVC(C= 1, kernel='rbf')
dt = DecisionTreeClassifier(random_state=42)
gbc = GradientBoostingClassifier(random_state=42)
rfc = RandomForestClassifier(n_estimators=100, random_state=42)
knn = KNeighborsClassifier(n_neighbors=5)
logistic = LogisticRegression(C = 1, solver = 'lbfgs')
linear = LinearRegression()
dummy = GaussianNB()

In [67]:
models = {
    'SVM': svc,
    'Logistic Regression': logistic,
    'KNN': knn,
    'GaussianNB': dummy,
    'Decision Tree': dt,
    'Random Forest': rfc,
    'GradientBoost': gbc,
}

In [68]:
train_accuracies = {}
val_accuracies = {}

train_f1s = {}
val_f1s = {}

In [69]:
for name, model in models.items():

    model.fit(x_train, y_train)

    y_train_pred = model.predict(x_train)
    y_val_pred = model.predict(x_val)

    train_accuracy = accuracy_score(y_train, y_train_pred)
    val_accuracy = accuracy_score(y_val, y_val_pred)

    train_f1 = f1_score(y_train, y_train_pred, average='weighted')
    val_f1 = f1_score(y_val, y_val_pred, average='weighted')

    train_accuracies[f'{name}'] = train_accuracy
    val_accuracies[f'{name}'] = val_accuracy

    train_f1s[f'{name}'] = train_f1
    val_f1s[f'{name}'] = val_f1

    print(name)
    print('Classification Report for train')
    print(classification_report(y_train, y_train_pred))
    print('Classification Report for validation')
    print(classification_report(y_val, y_val_pred))

    print('*'*50)


SVM
Classification Report for train
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        33
           1       0.00      0.00      0.00        71
           2       0.70      1.00      0.82       240

    accuracy                           0.70       344
   macro avg       0.23      0.33      0.27       344
weighted avg       0.49      0.70      0.57       344

Classification Report for validation
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        21
           1       0.00      0.00      0.00        48
           2       0.70      1.00      0.82       161

    accuracy                           0.70       230
   macro avg       0.23      0.33      0.27       230
weighted avg       0.49      0.70      0.58       230

**************************************************
Logistic Regression
Classification Report for train
              precision    recall  f1-score   support

       

/Users/laviniarotellini/Desktop/Laurea/Tesi/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/laviniarotellini/Desktop/Laurea/Tesi/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/laviniarotellini/Desktop/Laurea/Tesi/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio

Random Forest
Classification Report for train
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        33
           1       1.00      1.00      1.00        71
           2       1.00      1.00      1.00       240

    accuracy                           1.00       344
   macro avg       1.00      1.00      1.00       344
weighted avg       1.00      1.00      1.00       344

Classification Report for validation
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        21
           1       0.54      0.15      0.23        48
           2       0.71      0.96      0.82       161

    accuracy                           0.70       230
   macro avg       0.42      0.37      0.35       230
weighted avg       0.61      0.70      0.62       230

**************************************************


/Users/laviniarotellini/Desktop/Laurea/Tesi/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/laviniarotellini/Desktop/Laurea/Tesi/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/laviniarotellini/Desktop/Laurea/Tesi/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio

GradientBoost
Classification Report for train
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        33
           1       1.00      0.94      0.97        71
           2       0.98      1.00      0.99       240

    accuracy                           0.99       344
   macro avg       0.99      0.98      0.99       344
weighted avg       0.99      0.99      0.99       344

Classification Report for validation
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        21
           1       0.52      0.23      0.32        48
           2       0.73      0.93      0.81       161

    accuracy                           0.70       230
   macro avg       0.42      0.38      0.38       230
weighted avg       0.62      0.70      0.64       230

**************************************************


In [70]:
results_df = pd.DataFrame({
    'Model': list(train_accuracies.keys()),
    'Train Accuracy': list(train_accuracies.values()),
    'Validation Accuracy': list(val_accuracies.values()),
    'Train F1score': list(train_f1s.values()),
    'Validation F1score': list(val_f1s.values())
})

In [71]:
results_df = results_df.sort_values(by='Validation F1score', ascending=False)
results_df

,Model,Train Accuracy,Validation Accuracy,Train F1score,Validation F1score
1,Logistic Regression,0.837209,0.665217,0.825029,0.641014
6,GradientBoost,0.988372,0.695652,0.988252,0.636486
2,KNN,0.735465,0.656522,0.715638,0.622631
5,Random Forest,1.000000,0.704348,1.000000,0.621971
0,SVM,0.697674,0.700000,0.573431,0.576471
4,Decision Tree,1.000000,0.578261,1.000000,0.575288
3,GaussianNB,0.514535,0.286957,0.548844,0.310911


##  Fase di validazione

In [72]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

#### KNN

In [73]:
knn_params = {
    'n_neighbors' : [3, 5, 10],
    'weights' : ['uniform', 'distance'],
    'algorithm' : ['auto', 'brute'],
    'leaf_size' : [10, 30, 50],
    'p': [1, 2, 5],
    'metric' : ['minkowski']
}

knn_grid = GridSearchCV(KNeighborsClassifier(),
                        knn_params,
                        cv = cv,
                        scoring = 'f1_macro',
                        verbose = 1)

In [74]:
knn_grid.fit(x_train, y_train)
print(knn_grid.best_params_)

Fitting 5 folds for each of 108 candidates, totalling 540 fits
{'algorithm': 'auto', 'leaf_size': 10, 'metric': 'minkowski', 'n_neighbors': 3, 'p': 1, 'weights': 'uniform'}


In [75]:
# Allenamento e performance sul training set
knn_model = KNeighborsClassifier(**knn_grid.best_params_)
knn_model.fit(x_train, y_train)
y_pred_train_knn = knn_model.predict(x_train)

print(f'Accuracy di KNN sul training: {accuracy_score(y_train, y_pred_train_knn)}')
print(f'F1 macro di KNN sul training: {f1_score(y_train, y_pred_train_knn, average = 'macro')}')

Accuracy di KNN sul training: 0.7936046511627907
F1 macro di KNN sul training: 0.6622989936715427


In [76]:
# Performance sul validation set
y_pred_val_knn = knn_model.predict(x_val)
accuracy_knn = accuracy_score(y_val, y_pred_val_knn)
f1_knn = f1_score(y_val, y_pred_val_knn, average='macro')

print(f'Accuracy KNN sul validation: {accuracy_knn}')
print(f'F1 Macro del KNN sul validation set: {f1_knn}')

print('Classification report per KNN')
print(classification_report(y_val, y_pred_val_knn))

Accuracy KNN sul validation: 0.6086956521739131
F1 Macro del KNN sul validation set: 0.38851753852652965
Classification report per KNN
              precision    recall  f1-score   support

           0       0.19      0.19      0.19        21
           1       0.26      0.19      0.22        48
           2       0.73      0.79      0.76       161

    accuracy                           0.61       230
   macro avg       0.39      0.39      0.39       230
weighted avg       0.58      0.61      0.59       230



In [77]:
# Salvataggio risultati
evaluation_results['KNN'] = {
    'accuracy' : "{:.2f}".format(accuracy_knn),
    'f1' : "{:.2f}".format(f1_knn)
}

### Logistic Reg

In [78]:
log_reg_params = [
    {
        'solver': ['lbfgs', 'newton-cg'],
        'C': [0.01, 0.1, 1],
        'class_weight': ['balanced'],
        'max_iter': [300, 1000],
        'penalty': ['l2']
    },
    {
        'solver': ['sag', 'saga'],
        'C': [0.01, 0.1, 1],
        'class_weight': ['balanced'],
        'max_iter': [1000, 5000],
        'penalty': ['l2']
    }
]


grid_log_reg = RandomizedSearchCV(LogisticRegression(),
                            log_reg_params, 
                            cv = 5, 
                            scoring = 'f1_macro',
                            verbose=1,
                            )

In [79]:
grid_log_reg.fit(x_train, y_train)
print(grid_log_reg.best_params_)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
{'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 1000, 'class_weight': 'balanced', 'C': 1}


In [80]:
log_reg_model = LogisticRegression(**grid_log_reg.best_params_, random_state=42)
log_reg_model.fit(x_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclas

In [81]:
joblib.dump(
    log_reg_model,
    model_path + 'log_reg_tre_city'+ '.pkl'
)

['/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/Modelli a tre anni con city/log_reg_tre_city.pkl']

In [82]:
#log_reg_model = joblib.load('/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/Modelli a tre anni con city/log_reg_tre_city.pkl')

In [83]:
y_pred_train = log_reg_model.predict(x_train)
print('Classification Report per Logistic Regression')
print(classification_report(y_train, y_pred_train))
print(f'Accuracy della Logistic Regression: {accuracy_score(y_train, y_pred_train)}')
print(f'F1 Score della Logistic Regression: {f1_score(y_train, y_pred_train, average="weighted")}')

Classification Report per Logistic Regression
              precision    recall  f1-score   support

           0       0.59      1.00      0.74        33
           1       0.65      0.92      0.76        71
           2       0.97      0.76      0.85       240

    accuracy                           0.81       344
   macro avg       0.74      0.89      0.78       344
weighted avg       0.87      0.81      0.82       344

Accuracy della Logistic Regression: 0.813953488372093
F1 Score della Logistic Regression: 0.8213972902098963


In [84]:
# Performance sul validation
y_pred_log_reg = log_reg_model.predict(x_val)
lr_accuracy = accuracy_score(y_val, y_pred_log_reg)
lr_f1 = f1_score(y_val, y_pred_log_reg, average="macro")


print('Classification Report per Logistic Regression')
print(classification_report(y_val, y_pred_log_reg))
print(f'Accuracy della Logistic Regression: {lr_accuracy}')
print(f'F1 Score della Logistic Regression: {lr_f1}')

Classification Report per Logistic Regression
              precision    recall  f1-score   support

           0       0.37      0.48      0.42        21
           1       0.40      0.69      0.51        48
           2       0.85      0.64      0.73       161

    accuracy                           0.63       230
   macro avg       0.54      0.60      0.55       230
weighted avg       0.71      0.63      0.66       230

Accuracy della Logistic Regression: 0.6347826086956522
F1 Score della Logistic Regression: 0.5516184760865612


In [85]:
# Salvataggio risultati
evaluation_results['Logistic Regression'] = {
    'accuracy' : "{:.2f}".format(lr_accuracy),
    'f1' : "{:.2f}".format(lr_f1)
}

### SVM

In [86]:
svm_params = {
    'kernel' : ['rbf', 'linear', 'poly', 'sigmoid'], 
    'degree' : [2,3,5],
    'C' : [0.001, 0.01, 0.1, 1],
    'gamma': ['scale', 'auto'], 
    'class_weight' : ['balanced'],
    'max_iter' : [100, 300, 500, 1000]
}

In [87]:
svm_grid = GridSearchCV(SVC(), 
                        svm_params,
                        cv = 5,
                        scoring = 'f1_macro',
                        verbose = 1)

In [88]:
svm_grid.fit(x_train, y_train)
print(svm_grid.best_params_)

Fitting 5 folds for each of 384 candidates, totalling 1920 fits
{'C': 1, 'class_weight': 'balanced', 'degree': 2, 'gamma': 'scale', 'kernel': 'linear', 'max_iter': 1000}


In [89]:
svm_model = SVC(**svm_grid.best_params_, probability=True, random_state=42)
svm_model.fit(x_train, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",2
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",True
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [90]:
joblib.dump(
    svm_model,
    model_path + '_svm_tesi_tre_city'+ '.pkl'
)
#svm_model = joblib.load(f'/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/Modelli a tre anni con city/_svm_tesi_tre_city.pkl')

['/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/Modelli a tre anni con city/_svm_tesi_tre_city.pkl']

In [91]:
y_pred_train = svm_model.predict(x_train)
print('Classification Report per SVClassifier')
print(classification_report(y_train, y_pred_train))
print(f'Accuracy di SVM: {accuracy_score(y_train, y_pred_train)}')
print(f'F1 Score di SVM: {f1_score(y_train, y_pred_train, average="weighted")}')

Classification Report per SVClassifier
              precision    recall  f1-score   support

           0       0.57      1.00      0.73        33
           1       0.66      0.94      0.77        71
           2       0.98      0.75      0.85       240

    accuracy                           0.81       344
   macro avg       0.73      0.90      0.78       344
weighted avg       0.87      0.81      0.82       344

Accuracy di SVM: 0.813953488372093
F1 Score di SVM: 0.8218077630607719


In [92]:
# Performance sul validation set
y_pred_svm = svm_model.predict(x_val)
svm_accuracy = accuracy_score(y_val, y_pred_svm)
svm_f1 = f1_score(y_val, y_pred_svm, average="macro")


print('Classification Report per SVClassifier')
print(classification_report(y_val, y_pred_svm))
print(f'Accuracy di SVM: {svm_accuracy}')
print(f'F1 Score di SVM: {svm_f1}')

Classification Report per SVClassifier
              precision    recall  f1-score   support

           0       0.35      0.52      0.42        21
           1       0.39      0.67      0.49        48
           2       0.84      0.61      0.71       161

    accuracy                           0.61       230
   macro avg       0.53      0.60      0.54       230
weighted avg       0.70      0.61      0.64       230

Accuracy di SVM: 0.6130434782608696
F1 Score di SVM: 0.5397359229447857


In [93]:
# Salvataggio risultati
evaluation_results['SVM'] = {
    'accuracy' : "{:.2f}".format(svm_accuracy),
    'f1' : "{:.2f}".format(svm_f1)
}

### Gradient Boosting

In [94]:
gboost_params = {
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [50, 100, 200],
    'max_depth': [2, 3, 4],           
    'min_samples_leaf': [10, 20, 30],
    'subsample': [0.6, 0.8, 1.0],    
    'ccp_alpha': [0.01, 0.1, 0.5]
}

gboost_grid = GridSearchCV(GradientBoostingClassifier(),
                           gboost_params, 
                           cv = 5, 
                           scoring='f1_macro',
                           n_jobs=-1,
                           verbose=1)

gboost_grid.fit(x_train, y_train)
print(gboost_grid.best_params_)

Fitting 5 folds for each of 729 candidates, totalling 3645 fits
{'ccp_alpha': 0.01, 'learning_rate': 0.1, 'max_depth': 3, 'min_samples_leaf': 20, 'n_estimators': 200, 'subsample': 0.6}


In [95]:
gb_model = GradientBoostingClassifier(**gboost_grid.best_params_, random_state=42)
gb_model.fit(x_train, y_train)

,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",200
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",0.6
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are'friedman_mse' for the mean squared error with improvement score byFriedman, 'squared_error' for mean squared error. The default value of'friedman_mse' is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",20
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``,

In [96]:
joblib.dump(
    gb_model,
    model_path + '_gb_tesi_tre_city'+ '.pkl'
)
#gb_model = joblib.load(f'/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/Modelli a tre anni con city/_gb_tesi_tre_city.pkl')

['/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/Modelli a tre anni con city/_gb_tesi_tre_city.pkl']

In [97]:
y_pred_train = gb_model.predict(x_train)
print('Classification Report per Gradient Boosting Classifier')
print(classification_report(y_train, y_pred_train))
print(f'Accuracy del GBClassifier: {accuracy_score(y_train, y_pred_train)}')
print(f'F1 Score del GBClassifier: {f1_score(y_train, y_pred_train, average="weighted")}')

Classification Report per Gradient Boosting Classifier
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        33
           1       0.78      0.10      0.17        71
           2       0.71      0.99      0.83       240

    accuracy                           0.71       344
   macro avg       0.50      0.36      0.33       344
weighted avg       0.66      0.71      0.61       344

Accuracy del GBClassifier: 0.7122093023255814
F1 Score del GBClassifier: 0.6136722699696663


/Users/laviniarotellini/Desktop/Laurea/Tesi/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/laviniarotellini/Desktop/Laurea/Tesi/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/laviniarotellini/Desktop/Laurea/Tesi/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio

In [98]:
# Performance on the validation set
y_pred_gb = gb_model.predict(x_val)
gb_accuracy = accuracy_score(y_val, y_pred_gb)
gb_f1 = f1_score(y_val, y_pred_gb, average="macro")

print('Classification Report per Gradient Boosting Classifier')
print(classification_report(y_val, y_pred_gb))
print(f'Accuracy del GBClassifier: {gb_accuracy}')
print(f'F1 Score del GBClassifier: {gb_f1}')

Classification Report per Gradient Boosting Classifier
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        21
           1       0.33      0.02      0.04        48
           2       0.70      0.99      0.82       161

    accuracy                           0.70       230
   macro avg       0.34      0.34      0.29       230
weighted avg       0.56      0.70      0.58       230

Accuracy del GBClassifier: 0.6956521739130435
F1 Score del GBClassifier: 0.2862677717134964


/Users/laviniarotellini/Desktop/Laurea/Tesi/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/laviniarotellini/Desktop/Laurea/Tesi/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/laviniarotellini/Desktop/Laurea/Tesi/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio

In [99]:
# Salvataggio performance
evaluation_results['Gradient Boosting'] = {
    'accuracy' : "{:.2f}".format(gb_accuracy),
    'f1' : "{:.2f}".format(gb_f1)
}

### Random Forest

In [100]:
rf_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10],         
    'min_samples_split': [5, 10, 20], 
    'min_samples_leaf': [4, 8, 16],   
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced']
}

rf_grid = GridSearchCV(RandomForestClassifier(),
                       rf_params, 
                       cv = 5,
                       verbose = 1,
                       n_jobs = -1, 
                       scoring = 'f1_macro')

In [101]:
rf_grid.fit(x_train, y_train)
print(rf_grid.best_params_)

Fitting 5 folds for each of 162 candidates, totalling 810 fits
{'class_weight': 'balanced', 'max_depth': 10, 'max_features': 'log2', 'min_samples_leaf': 4, 'min_samples_split': 5, 'n_estimators': 100}


In [102]:
rf_model = RandomForestClassifier(**rf_grid.best_params_, random_state=42)
rf_model.fit(x_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",4
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'log2'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [103]:
joblib.dump(
    rf_model,
    model_path + '_rf_tesi_tre_city'+ '.pkl'
)
#rf_model = joblib.load(f'/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/Modelli a tre anni con city/_rf_tesi_tre_city.pkl')

['/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/Modelli a tre anni con city/_rf_tesi_tre_city.pkl']

In [104]:
y_pred_train = rf_model.predict(x_train)
print('Classification Report per Random Forest')
print(classification_report(y_train, y_pred_train))
print(f'Accuracy del Random Forest: {accuracy_score(y_train, y_pred_train)}')
print(f'F1 Score del Random Forest: {f1_score(y_train, y_pred_train, average="weighted")}')

Classification Report per Random Forest
              precision    recall  f1-score   support

           0       0.35      1.00      0.52        33
           1       0.57      0.76      0.65        71
           2       0.94      0.61      0.74       240

    accuracy                           0.68       344
   macro avg       0.62      0.79      0.64       344
weighted avg       0.81      0.68      0.70       344

Accuracy del Random Forest: 0.6773255813953488
F1 Score del Random Forest: 0.6993954376210373


In [105]:
# Performance sul validation set
y_pred_rf = rf_model.predict(x_val)
rf_accuracy = accuracy_score(y_val, y_pred_rf)
rf_f1 = f1_score(y_val, y_pred_rf, average="macro")

print('Classification Report per Random Forest')
print(classification_report(y_val, y_pred_rf))
print(f'Accuracy del Random Forest: {rf_accuracy}')
print(f'F1 Score del Random Forest: {rf_f1}')

Classification Report per Random Forest
              precision    recall  f1-score   support

           0       0.24      0.67      0.35        21
           1       0.37      0.48      0.42        48
           2       0.86      0.58      0.70       161

    accuracy                           0.57       230
   macro avg       0.49      0.58      0.49       230
weighted avg       0.70      0.57      0.61       230

Accuracy del Random Forest: 0.5695652173913044
F1 Score del Random Forest: 0.4881593714927048


In [106]:
# Salvataggio risultati
evaluation_results['Random Forest'] = {
    'accuracy' : "{:.2f}".format(rf_accuracy),
    'f1' : "{:.2f}".format(rf_f1)
}

### Decision Tree

In [107]:
dt_params = {
    'criterion' : ['gini', 'entropy', 'log_loss'],
    'max_depth' : [2, 4, 5],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced'],
    'max_features' : ['sqrt', 'log2']
}

dt_grid = GridSearchCV(DecisionTreeClassifier(), 
                       dt_params,
                       cv = 5, 
                       verbose = 1,
                       n_jobs = -1,
                       scoring = 'f1_macro')

In [108]:
dt_grid.fit(x_train, y_train)
print(dt_grid.best_params_)

Fitting 5 folds for each of 162 candidates, totalling 810 fits
{'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2}


In [109]:
dt_model = DecisionTreeClassifier(**dt_grid.best_params_, random_state=42)
dt_model.fit(x_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",'sqrt'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current 

In [110]:
joblib.dump(
    dt_model,
    model_path + '_dt_tesi_tre_city'+ '.pkl'
)
#dt_model = joblib.load(f'/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/Modelli a tre anni con city/_dt_tesi_tre_city.pkl')

['/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/Modelli a tre anni con city/_dt_tesi_tre_city.pkl']

In [111]:
y_pred_train = dt_model.predict(x_train)
print('Classification Report per Decision Tree')
print(classification_report(y_train, y_pred_train))
print(f'Accuracy del Decision Tree: {accuracy_score(y_train, y_pred_train)}')
print(f'F1 Score del Decision Tree: {f1_score(y_train, y_pred_train, average="weighted")}')

Classification Report per Decision Tree
              precision    recall  f1-score   support

           0       0.34      0.82      0.48        33
           1       0.39      0.79      0.52        71
           2       0.93      0.47      0.63       240

    accuracy                           0.57       344
   macro avg       0.55      0.69      0.54       344
weighted avg       0.76      0.57      0.59       344

Accuracy del Decision Tree: 0.5697674418604651
F1 Score del Decision Tree: 0.5905408961528097


In [112]:
# Performance sul validation set
y_pred_dt = dt_model.predict(x_val)
dt_accuracy = accuracy_score(y_val, y_pred_dt)
dt_f1 = f1_score(y_val, y_pred_dt, average="macro")

print('Classification Report per Decision Tree')
print(classification_report(y_val, y_pred_dt))
print(f'Accuracy del Decision Tree: {dt_accuracy}')
print(f'F1 Macro del Decision Tree: {dt_f1}')

Classification Report per Decision Tree
              precision    recall  f1-score   support

           0       0.29      0.62      0.39        21
           1       0.30      0.65      0.41        48
           2       0.81      0.41      0.55       161

    accuracy                           0.48       230
   macro avg       0.47      0.56      0.45       230
weighted avg       0.66      0.48      0.50       230

Accuracy del Decision Tree: 0.4782608695652174
F1 Macro del Decision Tree: 0.4490962254120148


In [113]:
evaluation_results['Decision Tree'] = {
    'accuracy' : "{:.2f}".format(dt_accuracy),
    'f1' : "{:.2f}".format(dt_f1)
}

### XGBoost

In [114]:
xgb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'gamma': [0, 1],
    'reg_alpha': [0, 0.1],
    'reg_lambda': [1, 5]
}

xgb_grid = GridSearchCV(XGBClassifier(), 
                       xgb_params,
                       cv = 5, 
                       verbose = 1,
                       n_jobs = -1,
                       scoring = 'f1_macro')

In [115]:
xgb_grid.fit(x_train, y_train)
print(xgb_grid.best_params_)

Fitting 5 folds for each of 864 candidates, totalling 4320 fits
{'colsample_bytree': 1.0, 'gamma': 0, 'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 300, 'reg_alpha': 0, 'reg_lambda': 1, 'subsample': 0.8}


In [116]:
xgb_model = XGBClassifier(**xgb_grid.best_params_, random_state = 42)
xgb_model.fit(x_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,1.0
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [117]:
joblib.dump(
    xgb_model,
    model_path + '_xgb_tesi_tre_city'+ '.pkl'
)
#xgb_model = joblib.load(f'{model_path}_xgb_tesi.pkl')

['/Users/laviniarotellini/Library/CloudStorage/OneDrive-UniversityofPisa(2)/models-tesi/Modelli a tre anni con city/_xgb_tesi_tre_city.pkl']

In [118]:
y_pred_xgb_train = xgb_model.predict(x_train)
print('Classification Report per XGBoost')
print(classification_report(y_train, y_pred_xgb_train))
print(f'Accuracy di XGBoost: {accuracy_score(y_train, y_pred_xgb_train)}')
print(f'F1 Score di XGBoost: {f1_score(y_train, y_pred_xgb_train, average="weighted")}')

Classification Report per XGBoost
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        33
           1       1.00      1.00      1.00        71
           2       1.00      1.00      1.00       240

    accuracy                           1.00       344
   macro avg       1.00      1.00      1.00       344
weighted avg       1.00      1.00      1.00       344

Accuracy di XGBoost: 1.0
F1 Score di XGBoost: 1.0


In [119]:
# Performance sul validation set
y_pred_xgb = xgb_model.predict(x_val)
xgb_accuracy = accuracy_score(y_val, y_pred_xgb)
xgb_f1 = f1_score(y_val, y_pred_xgb, average="macro")

print('Classification Report per XGBoost')
print(classification_report(y_val, y_pred_xgb))
print(f'Accuracy di XGBoost: {xgb_accuracy}')
print(f'F1 Score di XGBoost: {xgb_f1}')

Classification Report per XGBoost
              precision    recall  f1-score   support

           0       0.20      0.05      0.08        21
           1       0.30      0.25      0.27        48
           2       0.71      0.82      0.76       161

    accuracy                           0.63       230
   macro avg       0.40      0.37      0.37       230
weighted avg       0.58      0.63      0.60       230

Accuracy di XGBoost: 0.6304347826086957
F1 Score di XGBoost: 0.37088537666572347


In [120]:
# Salvataggio risultati
evaluation_results['XGBoost'] = {
    'accuracy' : "{:.2f}".format(xgb_accuracy),
    'f1' : "{:.2f}".format(xgb_f1)
}

### Recap Validation Phas

In [121]:
validation_df =  pd.DataFrame({'Model': list(evaluation_results.keys()),
                               'Results' : list(evaluation_results.values())})

In [122]:
validation_df

,Model,Results
0,KNN,"{'accuracy': '0.61', 'f1': '0.39'}"
1,Logistic Regression,"{'accuracy': '0.63', 'f1': '0.55'}"
2,SVM,"{'accuracy': '0.61', 'f1': '0.54'}"
3,Gradient Boosting,"{'accuracy': '0.70', 'f1': '0.29'}"
4,Random Forest,"{'accuracy': '0.57', 'f1': '0.49'}"
5,Decision Tree,"{'accuracy': '0.48', 'f1': '0.45'}"
6,XGBoost,"{'accuracy': '0.63', 'f1': '0.37'}"


In [123]:
## Unificazione training set
x_train_final = pd.concat([x_train, x_val])
y_train_final  = pd.concat([y_train, y_val])

In [124]:
## Ri allenamento dei modelli
### KNN ###
knn_model_final = KNeighborsClassifier(**knn_grid.best_params_)
knn_model_final.fit(x_train_final, y_train_final)

### LOGISTIC REGRESSION ###
log_reg_model_final = LogisticRegression(**grid_log_reg.best_params_, random_state=42)
log_reg_model_final.fit(x_train_final, y_train_final)

### SVM ###
svm_model_final = SVC(**svm_grid.best_params_, random_state=42)
svm_model_final.fit(x_train_final, y_train_final)

### RANDOM FOREST ###
rf_model_final = RandomForestClassifier(**rf_grid.best_params_, random_state=42)
rf_model_final.fit(x_train_final, y_train_final)

### XGBOOST ###
xgb_model_final = XGBClassifier(**xgb_grid.best_params_, random_state = 42)
xgb_model_final.fit(x_train_final, y_train_final)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,1.0
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

## Evaluation

In [125]:
y_test = df_test['target']
x_test = df_test.drop('target', axis=1)

In [127]:
lab_encoder = joblib.load('/Users/laviniarotellini/Desktop/Laurea/Tesi/Costruzione e Analisi DF/3. Data Understanding and Prep/label_encoder.pkl')

In [128]:
# Tutti i modelli con i rispettivi predittori
trained_models = {
    'Logistic Regression': (log_reg_model, y_pred_log_reg),
    'SVM': (svm_model, y_pred_svm),
    'Gradient Boosting': (gb_model, y_pred_gb),
    'Random Forest': (rf_model, y_pred_rf),
    'Decision Tree': (dt_model, y_pred_dt),
    'XGBoost': (xgb_model, y_pred_xgb),
}

test_results = []

for name, (model, _) in trained_models.items():
    y_test_pred = model.predict(x_test)
    
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_f1_weighted = f1_score(y_test, y_test_pred, average='weighted')
    test_f1_macro = f1_score(y_test, y_test_pred, average='macro')
    
    test_results.append({
        'Model': name,
        'Test Accuracy': test_accuracy,
        'Test F1 Weighted': test_f1_weighted,
        'Test F1 Macro': test_f1_macro,
    })
    
    print(f'\n{name}')
    print(classification_report(y_test, y_test_pred, 
                                  target_names=lab_encoder.classes_))
    print('*' * 50)

test_results_df = pd.DataFrame(test_results).sort_values(
    by='Test F1 Macro', ascending=False
)
print(test_results_df)


Logistic Regression
              precision    recall  f1-score   support

      inflow       0.31      0.47      0.37        38
     outflow       0.38      0.47      0.42        90
      stable       0.78      0.65      0.71       256

    accuracy                           0.59       384
   macro avg       0.49      0.53      0.50       384
weighted avg       0.64      0.59      0.61       384

**************************************************

SVM
              precision    recall  f1-score   support

      inflow       0.26      0.39      0.31        38
     outflow       0.37      0.49      0.42        90
      stable       0.79      0.64      0.70       256

    accuracy                           0.58       384
   macro avg       0.47      0.51      0.48       384
weighted avg       0.64      0.58      0.60       384

**************************************************

Gradient Boosting
              precision    recall  f1-score   support

      inflow       0.00      0.00   

/Users/laviniarotellini/Desktop/Laurea/Tesi/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/laviniarotellini/Desktop/Laurea/Tesi/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/laviniarotellini/Desktop/Laurea/Tesi/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio